# 示例1：大模型分析工具的调用

In [2]:
# 1、获取大模型
#1.导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
dotenv.load_dotenv()
from langchain_openai import ChatOpenAI
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 定义LLM模型
chat_model =ChatOpenAI(model="gpt-4o-mini",temperature=0)

# 2、获取工具的列表
tools = [MoveFileTool()]
# 3、这里需要将工具转换为openai函数，后续再将函数传入模型调用
functions = [convert_to_openai_function(t) for t in tools]
# 4、获取消息列表
messages = [HumanMessage(content = "将文件a移动到桌面")]
# 5、调用大模型（传入消息列表、工具列表）
response = chat_model.invoke(input=messages,functions=functions) # invoke调用要传入函数的列表
print(response)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"a","destination_path":"/Users/YourUsername/Desktop/a"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 76, 'total_tokens': 103, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DNdxe4MRT0yQJHqOwZyQVJqYgOO3e', 'finish_reason': 'function_call', 'logprobs': None} id='lc_run--019d2a15-0d6f-7192-a379-f8a9e8991d2e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 76, 'output_tokens': 27, 'total_tokens': 103, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [3]:
functions = [convert_to_openai_function(t) for t in tools]
# 4、获取消息列表
messages = [HumanMessage(content = "明天天气怎么样")]
# 5、调用大模型（传入消息列表、工具列表）
response = chat_model.invoke(input=messages,functions=functions) # invoke调用要传入函数的列表
print(response)

content='抱歉，我无法提供实时天气信息。建议您查看天气预报网站或使用天气应用程序获取最新的天气情况。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 73, 'total_tokens': 102, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DNeD9bbudmYODbuBSAQJorHJwvA5f', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d2a23-bb49-7aa0-8e9e-d85a7310e944-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 73, 'output_tokens': 29, 'total_tokens': 102, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


通过以上两个案例，能够得到AIMessage的核心属性如下：

1、如果分析出需要调用对应的工具：

content：信息为空，因为大模型要调用工具，因此不会直接返回信息给用户

additional_kwargs：包含function_call字段，指明具体函数调用的参数和函数名

2、如果分析出不需要调用对应的工具：

content:信息不为空

additional_kwargs:不包含调用字段

# 示例2：如何调用具体大模型分析出来的工具

针对大模型：仅能分析出要调用的工具，此工具无法真正的执行，Agent不但可以分析出需要的工具，还能执行

In [8]:
# 1、获取大模型
#1.导入相关依赖
from langchain_community.tools import MoveFileTool
from langchain_core.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_function
import os
import dotenv
dotenv.load_dotenv()
from langchain_openai import ChatOpenAI
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 定义LLM模型
chat_model =ChatOpenAI(model="gpt-4o-mini",temperature=0)

# 2、获取工具的列表
tools = [MoveFileTool()]
# 3、这里需要将工具转换为openai函数，后续再将函数传入模型调用
functions = [convert_to_openai_function(t) for t in tools]
# 4、获取消息列表
messages = [HumanMessage(content = "将当前目录下的a.txt文件移动到C:\\桌面")]
# 5、调用大模型（传入消息列表、工具列表）
response = chat_model.invoke(input=messages,functions=functions) # invoke调用要传入函数的列表


步骤1：分析需要调用哪个工具或函数

In [12]:
import json
if "function_call" in response.additional_kwargs:
    tool_name = response.additional_kwargs["function_call"]["name"]
    tool_args = json.loads(response.additional_kwargs["function_call"]["arguments"])
    print(f"调用工具：{tool_name} \n 参数：{tool_args}")

else:
    print(f"模型回复：{response.content}")

调用工具：move_file 
 参数：{'source_path': 'a.txt', 'destination_path': 'C:\\桌面\\a.txt'}


步骤2：调用对应的工具

In [13]:
if "move_file" in response.additional_kwargs["function_call"]["name"]:
    tool = MoveFileTool()
    result = tool.run(tool_args) #调用工具
    print(result)

File moved successfully from a.txt to C:\桌面\a.txt.
